# RESNET-50 OPTIMIZED TRAINING FOR RETINAL DISEASE DETECTION

```
Hardware     : HP Omen 16 (RTX 4060 8GB, i7-13700HX)
Dataset      : RFMiD (3200 images, 46 diseases)
Optimizations: Mixed Precision (BF16/FP16), TF32, Optimised Data Loading
Tested on    : torch==2.1.0
```

## 1. Imports

In [3]:
import os, gc, time, json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms

# Version-safe mixed-precision imports.
# GradScaler was promoted to torch.amp only in PyTorch >= 2.3.
# On 2.1 / 2.2 it still lives in torch.cuda.amp.
_pt_ver = tuple(int(x) for x in torch.__version__.split('.')[:2])
if _pt_ver >= (2, 3):
    from torch.amp import autocast, GradScaler
    def make_scaler(): return GradScaler('cuda')
else:
    from torch.cuda.amp import autocast, GradScaler
    def make_scaler(): return GradScaler()

from sklearn.metrics import f1_score, precision_recall_fscore_support

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')

PyTorch version : 2.1.0+cu121
CUDA available  : True


In [4]:
import torch

print(f"PyTorch        : {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
print(f"GPU            : {torch.cuda.get_device_name(0)}")
print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"BF16 support   : {torch.cuda.is_bf16_supported()}")

# Quick sanity test - multiply two tensors on GPU
a = torch.randn(1000, 1000, device='cuda')
b = torch.randn(1000, 1000, device='cuda')
c = a @ b
print(f"GPU tensor test: PASSED  (result shape: {c.shape})")

PyTorch        : 2.1.0+cu121
CUDA available : True
GPU            : NVIDIA GeForce RTX 4060 Laptop GPU
VRAM           : 8.59 GB
BF16 support   : True
GPU tensor test: PASSED  (result shape: torch.Size([1000, 1000]))


## 2. Hardware Optimization (RTX 4060 Specific)

In [5]:
class HardwareOptimizer:
    """Optimise for HP Omen 16 RTX 4060."""

    def __init__(self):
        self.device       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.gpu_name     = 'N/A'
        self.vram_total   = 0.0
        self.cuda_version = 'N/A'
        self.use_bf16     = False

        if self.device.type == 'cuda':
            self.gpu_name     = torch.cuda.get_device_name(0)
            self.vram_total   = torch.cuda.get_device_properties(0).total_memory / 1e9
            self.cuda_version = torch.version.cuda

            torch.backends.cudnn.benchmark        = True
            torch.backends.cuda.matmul.allow_tf32 = True
            torch.backends.cudnn.allow_tf32        = True

            self.use_bf16 = torch.cuda.is_bf16_supported()

    def print_specs(self):
        print('=' * 60)
        print('HP Omen 16 Hardware Configuration')
        print('=' * 60)
        print('CPU: Intel i7-13700HX (16 cores)')
        print(f'GPU: {self.gpu_name}')
        if self.device.type == 'cuda':
            print(f'VRAM         : {self.vram_total:.1f} GB')
            print(f'CUDA         : {self.cuda_version}')
            print(f'BF16 Support : {self.use_bf16}')
            print('TF32 Enabled : True')
        print('=' * 60)


hw       = HardwareOptimizer()
hw.print_specs()

device   = hw.device
use_bf16 = hw.use_bf16


def set_seed(seed: int = 42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

HP Omen 16 Hardware Configuration
CPU: Intel i7-13700HX (16 cores)
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM         : 8.6 GB
CUDA         : 12.1
BF16 Support : True
TF32 Enabled : True


## 3. Optimized Dataset Class

In [6]:
class OptimizedRFMiDDataset(Dataset):
    """Memory-efficient dataset with PIL loading."""

    def __init__(self, img_dir, labels_df, transform=None):
        self.img_dir       = img_dir
        self.labels_df     = labels_df.reset_index(drop=True)
        self.transform     = transform
        self.disease_cols  = [c for c in labels_df.columns if c != 'ID']
        self.num_classes   = len(self.disease_cols)
        self.img_paths     = []
        self.valid_indices = []

        for idx, img_name in enumerate(self.labels_df['ID'].values):
            base  = os.path.join(img_dir, str(img_name))
            found = False
            for ext in ['', '.png', '.jpg', '.jpeg']:
                if os.path.exists(base + ext):
                    self.img_paths.append(base + ext)
                    self.valid_indices.append(idx)
                    found = True
                    break
            if not found:
                print(f'Warning: image not found -> {img_name}')
                self.img_paths.append(None)
                self.valid_indices.append(idx)

        print(f'Loaded {len(self.valid_indices)} images from {img_dir}')

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        p = self.img_paths[idx]
        if p and os.path.exists(p):
            try:
                image = Image.open(p).convert('RGB')
            except Exception:
                image = Image.new('RGB', (224, 224), 'black')
        else:
            image = Image.new('RGB', (224, 224), 'black')

        if self.transform:
            image = self.transform(image)

        orig  = self.valid_indices[idx]
        lbl   = self.labels_df.iloc[orig][self.disease_cols].values.astype(np.float32)
        return image, torch.tensor(lbl, dtype=torch.float32)

    def get_class_weights(self):
        counts = self.labels_df[self.disease_cols].sum().values
        w = len(self.labels_df) / (self.num_classes * (counts + 1e-6))
        return torch.tensor(w, dtype=torch.float32)

## 4. Optimized Transforms

In [8]:
class OptimizedTransforms:
    _MEAN = [0.485, 0.456, 0.406]
    _STD  = [0.229, 0.224, 0.225]

    @classmethod
    def get_train_transform(cls, sz=224):
        return transforms.Compose([
            transforms.Resize((sz, sz)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.3),
            transforms.RandomRotation(degrees=30),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
            transforms.ToTensor(),
            transforms.Normalize(cls._MEAN, cls._STD),
        ])

    @classmethod
    def get_val_transform(cls, sz=224):
        return transforms.Compose([
            transforms.Resize((sz, sz)),
            transforms.ToTensor(),
            transforms.Normalize(cls._MEAN, cls._STD),
        ])

## 5. Load Dataset

In [9]:
import os
os.chdir(r'D:\OcuSight')   # ← add this line

DATA_DIR     = 'data/rfmid'
TRAIN_IMAGES = os.path.join(DATA_DIR, 'images', 'train')
VAL_IMAGES   = os.path.join(DATA_DIR, 'images', 'val')
TEST_IMAGES  = os.path.join(DATA_DIR, 'images', 'test')
TRAIN_LABELS = os.path.join(DATA_DIR, 'labels', 'train_labels.csv')
VAL_LABELS   = os.path.join(DATA_DIR, 'labels', 'val_labels.csv')
TEST_LABELS  = os.path.join(DATA_DIR, 'labels', 'test_labels.csv')

print(f'Working dir: {os.getcwd()}')   # confirm it's D:\OcuSight
print('Loading datasets...')
train_df = pd.read_csv(TRAIN_LABELS)
val_df   = pd.read_csv(VAL_LABELS)
test_df  = pd.read_csv(TEST_LABELS)

print(f'Training   : {len(train_df)} samples')
print(f'Validation : {len(val_df)} samples')
print(f'Testing    : {len(test_df)} samples')

disease_cols = [c for c in train_df.columns if c != 'ID']
num_classes  = len(disease_cols)
print(f'Disease classes: {num_classes}')

train_dataset = OptimizedRFMiDDataset(TRAIN_IMAGES, train_df, OptimizedTransforms.get_train_transform())
val_dataset   = OptimizedRFMiDDataset(VAL_IMAGES,   val_df,   OptimizedTransforms.get_val_transform())
test_dataset  = OptimizedRFMiDDataset(TEST_IMAGES,  test_df,  OptimizedTransforms.get_val_transform())

Working dir: D:\OcuSight
Loading datasets...
Training   : 1920 samples
Validation : 640 samples
Testing    : 640 samples
Disease classes: 46
Loaded 1920 images from data/rfmid\images\train
Loaded 640 images from data/rfmid\images\val
Loaded 640 images from data/rfmid\images\test


## 6. Optimized DataLoaders

In [10]:
BATCH_SIZE = 32 
NUM_WORKERS = 0 
PIN_MEMORY = True
PREFETCH_FACTOR = None

_kw = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    prefetch_factor=PREFETCH_FACTOR,
    persistent_workers=False,  # False since num_workers=0
)

train_loader = DataLoader(train_dataset, shuffle=True, **_kw)
val_loader = DataLoader(val_dataset, shuffle=False, **_kw)
test_loader = DataLoader(test_dataset, shuffle=False, **_kw)

print(f'Batch size    : {BATCH_SIZE}')
print(f'Workers       : {NUM_WORKERS}')
print(f'Train batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')
print(f'Test batches  : {len(test_loader)}')

Batch size    : 32
Workers       : 0
Train batches : 60
Val batches   : 20
Test batches  : 20


## 7. Build ResNet-50 Model

In [11]:
def build_resnet50(num_classes: int) -> nn.Module:
    model = models.resnet50(weights='IMAGENET1K_V1')
    nf = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(nf, 512),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, num_classes),
    )
    return model


print('Building ResNet-50...')
model = build_resnet50(num_classes).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Total parameters     : {total_params:,}')
print(f'Trainable parameters : {trainable_params:,}')
print(f'Model size           : {total_params * 4 / 1024 / 1024:.2f} MB')

Building ResNet-50...
Total parameters     : 24,580,718
Trainable parameters : 24,580,718
Model size           : 93.77 MB


## 8. Training Components

In [13]:
# Keep class weights for imbalance handling
class_weights = train_dataset.get_class_weights().to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=class_weights)  # ← KEEP pos_weight

optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01, betas=(0.9, 0.999))  # lr=1e-4 is fine

NUM_EPOCHS = 50
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
scaler = None if use_bf16 else make_scaler()

print(f'Precision : {"BF16" if use_bf16 else "FP16"}')
print(f'LR        : 1e-4   Weight decay: 0.01')
print(f'Epochs    : {NUM_EPOCHS}')
print(f'Class weights: {class_weights is not None}')  # Verify class weights are loaded

Precision : BF16
LR        : 1e-4   Weight decay: 0.01
Epochs    : 50
Class weights: True


## 9. Training Functions

In [14]:
def train_epoch(model, loader, criterion, optimizer, scaler, device, amp_dtype):
    model.train()
    total_loss, n = 0.0, 0

    for images, labels in tqdm(loader, desc='Training', leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        # FIXED: No device_type argument for PyTorch 2.1
        with autocast(dtype=amp_dtype):
            loss = criterion(model(images), labels)

        if scaler:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        total_loss += loss.item()
        n += 1

    return total_loss / n


def validate_epoch(model, loader, criterion, device, amp_dtype):
    model.eval()
    total_loss, n = 0.0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Validating', leave=False):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            # FIXED: No device_type argument for PyTorch 2.1
            with autocast(dtype=amp_dtype):
                outputs = model(images)
                loss = criterion(outputs, labels)

            total_loss += loss.item()
            n += 1
            all_preds.append((torch.sigmoid(outputs) > 0.5).float().cpu())
            all_labels.append(labels.cpu())

    preds = torch.cat(all_preds).numpy()
    labels = torch.cat(all_labels).numpy()

    return {
        'loss': total_loss / n,
        'accuracy': float(np.mean(np.all(preds == labels, axis=1))),
        'f1_macro': f1_score(labels, preds, average='macro', zero_division=0),
        'f1_micro': f1_score(labels, preds, average='micro', zero_division=0),
        'predictions': preds,
        'labels': labels,
    }

## 10. Training Loop

In [15]:
NUM_EPOCHS = 50  # ← ADD THIS LINE FIRST

os.makedirs('checkpoints', exist_ok=True)
os.makedirs('outputs',     exist_ok=True)

history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1_macro': [], 'val_f1_micro': []}

best_f1            = 0.0
early_stop_counter = 0
PATIENCE           = 10

print('=' * 60)
print('STARTING RESNET-50 TRAINING')
print('=' * 60)
t0 = time.time()

for epoch in range(NUM_EPOCHS):
    te = time.time()
    print(f'\nEpoch {epoch + 1}/{NUM_EPOCHS}')
    print('-' * 40)

    train_loss  = train_epoch(model, train_loader, criterion, optimizer, scaler, device, amp_dtype)
    val_metrics = validate_epoch(model, val_loader, criterion, device, amp_dtype)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_metrics['loss'])
    history['val_acc'].append(val_metrics['accuracy'])
    history['val_f1_macro'].append(val_metrics['f1_macro'])
    history['val_f1_micro'].append(val_metrics['f1_micro'])

    scheduler.step()

    print(f'  Train Loss   : {train_loss:.4f}')
    print(f'  Val Loss     : {val_metrics["loss"]:.4f}')
    print(f'  Val Accuracy : {val_metrics["accuracy"]:.4f}')
    print(f'  Val F1 macro : {val_metrics["f1_macro"]:.4f}')
    print(f'  Val F1 micro : {val_metrics["f1_micro"]:.4f}')
    print(f'  LR           : {optimizer.param_groups[0]["lr"]:.6f}')
    print(f'  Time         : {time.time() - te:.1f}s')

    if val_metrics['f1_macro'] > best_f1:
        best_f1 = val_metrics['f1_macro']
        torch.save({
            'epoch'               : epoch,
            'model_state_dict'    : model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_f1'              : best_f1,
            'val_accuracy'        : val_metrics['accuracy'],
        }, 'checkpoints/best_resnet50.pth')
        print(f'  => New best model saved! F1: {best_f1:.4f}')
        early_stop_counter = 0
    else:
        early_stop_counter += 1

    if early_stop_counter >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch + 1}')
        break

    torch.cuda.empty_cache()
    gc.collect()

print(f'\nTraining complete in {(time.time() - t0) / 60:.2f} min')
print(f'Best Val F1 : {best_f1:.4f}')

STARTING RESNET-50 TRAINING

Epoch 1/50
----------------------------------------


  Train Loss   : 0.2158
  Val Loss     : 16801.7762
  Val Accuracy : 0.2094
  Val F1 macro : 0.0000
  Val F1 micro : 0.0000
  LR           : 0.000100
  Time         : 218.7s

Epoch 2/50
----------------------------------------


  Train Loss   : 0.1050
  Val Loss     : 16535.9319
  Val Accuracy : 0.2094
  Val F1 macro : 0.0000
  Val F1 micro : 0.0000
  LR           : 0.000100
  Time         : 224.6s

Epoch 3/50
----------------------------------------


KeyboardInterrupt: 

## 11. Plot Training Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0,0].plot(history['val_loss'],   label='Val',   linewidth=2)
axes[0,0].set(xlabel='Epoch', ylabel='Loss', title='Loss'); axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

axes[0,1].plot(history['val_acc'], color='green', linewidth=2)
axes[0,1].set(xlabel='Epoch', ylabel='Accuracy', title='Validation Accuracy'); axes[0,1].grid(alpha=0.3)

axes[1,0].plot(history['val_f1_macro'], color='orange', linewidth=2)
axes[1,0].set(xlabel='Epoch', ylabel='F1', title='Val F1 Macro'); axes[1,0].grid(alpha=0.3)

axes[1,1].plot(history['val_f1_micro'], color='red', linewidth=2)
axes[1,1].set(xlabel='Epoch', ylabel='F1', title='Val F1 Micro'); axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Evaluate on Test Set

In [ ]:
ckpt = torch.load('checkpoints/best_resnet50.pth', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
print(f'Loaded epoch {ckpt["epoch"] + 1}  |  Val F1: {ckpt["val_f1"]:.4f}')

test_metrics = validate_epoch(model, test_loader, criterion, device, amp_dtype)

print('\n' + '=' * 60)
print('TEST SET RESULTS')
print('=' * 60)
print(f'Test Loss     : {test_metrics["loss"]:.4f}')
print(f'Test Accuracy : {test_metrics["accuracy"]:.4f}')
print(f'Test F1 macro : {test_metrics["f1_macro"]:.4f}')
print(f'Test F1 micro : {test_metrics["f1_micro"]:.4f}')

## 13. Per-Class Performance

In [ ]:
precision, recall, f1, support = precision_recall_fscore_support(
    test_metrics['labels'], test_metrics['predictions'], average=None, zero_division=0
)

per_class_df = pd.DataFrame({
    'Disease'  : disease_cols,
    'Precision': precision,
    'Recall'   : recall,
    'F1-Score' : f1,
    'Samples'  : support,
}).sort_values('F1-Score', ascending=False)

per_class_df.to_csv('outputs/per_class_results.csv', index=False)
print('Top 10:')
print(per_class_df.head(10).to_string(index=False))
print('\nBottom 10:')
print(per_class_df.tail(10).to_string(index=False))

## 14. Visualize Per-Class Performance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

top10    = per_class_df.head(10)
bottom10 = per_class_df.tail(10)

axes[0].barh(top10['Disease'],    top10['F1-Score'],    color='green', alpha=0.7)
axes[0].set(xlabel='F1 Score', title='Top 10 Diseases', xlim=(0, 1)); axes[0].grid(alpha=0.3)

axes[1].barh(bottom10['Disease'], bottom10['F1-Score'], color='red',   alpha=0.7)
axes[1].set(xlabel='F1 Score', title='Bottom 10 Diseases (Rare)', xlim=(0, 1)); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/per_class_performance.png', dpi=150, bbox_inches='tight')
plt.show()

## 15. Save Results

In [ ]:
results = {
    'best_val_f1'  : best_f1,
    'test_accuracy': float(test_metrics['accuracy']),
    'test_f1_macro': float(test_metrics['f1_macro']),
    'test_f1_micro': float(test_metrics['f1_micro']),
    'history': {
        'train_loss'  : history['train_loss'],
        'val_loss'    : history['val_loss'],
        'val_accuracy': history['val_acc'],
        'val_f1_macro': history['val_f1_macro'],
    },
    'hardware': {
        'gpu'            : hw.gpu_name if device.type == 'cuda' else 'CPU',
        'batch_size'     : BATCH_SIZE,
        'mixed_precision': 'BF16' if use_bf16 else 'FP16',
    },
}

with open('outputs/training_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('Results saved to outputs/')

## 16. Summary Report

In [ ]:
print('=' * 60)
print('RESNET-50 TRAINING SUMMARY')
print('=' * 60)
print(f'\nDataset      : {len(train_dataset)} train / {len(val_dataset)} val / {len(test_dataset)} test  |  {num_classes} classes')
print(f'Model        : ResNet-50  ({total_params:,} params, {total_params*4/1024/1024:.1f} MB)')
print(f'Epochs run   : {len(history["train_loss"])}')
print(f'Precision    : {"BF16" if use_bf16 else "FP16"}')
print(f'\nBest Val F1  : {best_f1:.4f}')
print(f'Test F1 macro: {test_metrics["f1_macro"]:.4f}')
print(f'Test Accuracy: {test_metrics["accuracy"]:.4f}')
if device.type == 'cuda':
    print(f'\nGPU : {hw.gpu_name}  ({hw.vram_total:.1f} GB VRAM)')
print('\n' + '=' * 60)

## 17. Inference Helper

In [ ]:
def predict_image(image_path: str, model: nn.Module, device, amp_dtype):
    """Return predicted diseases and top probabilities for one image."""
    tensor = OptimizedTransforms.get_val_transform()(Image.open(image_path).convert('RGB'))
    tensor = tensor.unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        with autocast(device_type='cuda', dtype=amp_dtype):
            probs = torch.sigmoid(model(tensor)).cpu().numpy()[0]

    positive  = [disease_cols[i] for i, p in enumerate(probs) if p > 0.5]
    top_probs = {disease_cols[i]: float(probs[i]) for i in range(len(probs)) if probs[i] > 0.3}
    return positive, top_probs


print('predict_image() is ready.')
print('Usage: diseases, probs = predict_image("path/to/image.png", model, device, amp_dtype)')
# diseases, probs = predict_image('data/rfmid/images/test/1.png', model, device, amp_dtype)
# print(diseases, probs)